In [60]:
import pandas as pd

In [61]:
df= pd.read_excel(r"C:\Users\Monsruti Devi\Documents\BRUCEA MOLLIS QSAR\book2.xlsx")
df

,0,CHEMICAL CONSTITUENTS
0,1,Bruceine A
1,2,Bruceine B
2,3,Bruceine C
3,4,Bruceine D
4,5,Bruceine E
...,...,...
92,93,Brujavanone J
93,94,Brujavanone K
94,95,Brujavanone L
95,96,Brujavanone M


In [62]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

# Load data
df = pd.read_excel(r"C:\Users\Monsruti Devi\Documents\BRUCEA MOLLIS QSAR\book2.xlsx")

# Connect to molecule database
molecule = new_client.molecule

# Create empty lists to store results
chembl_ids = []
smiles_list = []

for compound in df['CHEMICAL CONSTITUENTS']:
    
    result = molecule.filter(pref_name__iexact=compound)
    
    if result:
        chembl_ids.append(result[0]['molecule_chembl_id'])
        
        if result[0]['molecule_structures']:
            smiles_list.append(result[0]['molecule_structures']['canonical_smiles'])
        else:
            smiles_list.append(None)
    else:
        chembl_ids.append(None)
        smiles_list.append(None)

# Add to dataframe
df['ChEMBL_ID'] = chembl_ids
df['SMILES'] = smiles_list

# Save
df.to_csv("brucea_with_smiles.csv", index=False)

print("Done 🌿")

Done 🌿


In [63]:
# Set options to display all rows and columns
df= pd.set_option('display.max_rows', None)
df= pd.set_option('display.max_columns', None)
df= pd.set_option('display.max_colwidth', None) # To prevent truncation of long text in cells

# Display your DataFrame
df



In [64]:
from chembl_webresource_client.new_client import new_client

activity = new_client.activity

res = activity.filter(
    molecule_chembl_id="CHEMBL12345",
    standard_type="IC50"
)

for r in res:
    print(r["standard_value"], r["standard_units"])

In [65]:
! pip install tqdm

In [66]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
from tqdm import tqdm

# connect to ChEMBL
activity = new_client.activity
molecule = new_client.molecule

# function to fetch IC50
def get_ic50(compound):
    try:
        mol = molecule.search(compound)
        chembl_id = mol[0]['molecule_chembl_id']

        res = activity.filter(
            molecule_chembl_id=chembl_id,
            standard_type="IC50"
        )

        if res:
            return pd.Series([res[0]['standard_value'], res[0]['standard_units']])
        else:
            return pd.Series([None, None])

    except:
        return pd.Series([None, None])

# enable progress bar
tqdm.pandas()

# apply with progress bar
df[['standard_value','units']] = df['CHEMICAL CONSTITUENTS'].progress_apply(get_ic50)

TypeError: 'NoneType' object is not subscriptable

In [ ]:
df

,0,CHEMICAL CONSTITUENTS,ChEMBL_ID,SMILES,standard_value,units
0,1,Bruceine A,CHEMBL250451,COC(=O)[C@@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)...,0.004,ug.mL-1
1,2,Bruceine B,CHEMBL140809,COC(=O)[C@@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)...,0.893,ug.mL-1
2,3,Bruceine C,CHEMBL455759,COC(=O)[C@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)[...,0.107,ug.mL-1
3,4,Bruceine D,CHEMBL477693,CC1=CC(=O)[C@@H](O)[C@]2(C)[C@H]3[C@@H](O)[C@H...,0.835,ug.mL-1
4,5,Bruceine E,CHEMBL2037038,CC1=C[C@H](O)[C@@H](O)[C@]2(C)[C@H]3[C@@H](O)[...,5000.0,nM
...,...,...,...,...,...,...
92,93,Brujavanone J,NaN,NaN,100000.0,nM
93,94,Brujavanone K,NaN,NaN,220.0,nM
94,95,Brujavanone L,NaN,NaN,10000.0,nM
95,96,Brujavanone M,NaN,NaN,4900.0,nM


In [ ]:
# convert IC50 values to numeric
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

# remove rows where IC50 is missing
df = df.dropna(subset=["standard_value"])

# function to classify bioactivity
def bioactivity_class(value):
    
    if value <= 1000:
        return "Active"
    
    elif value <= 10000:
        return "Intermediate"
    
    else:
        return "Inactive"

# create new column
df["bioactivity_class"] = df["standard_value"].apply(bioactivity_class)

# optional: reset index after cleaning
df = df.reset_index(drop=True)

In [ ]:
df

,0,CHEMICAL CONSTITUENTS,ChEMBL_ID,SMILES,standard_value,units,bioactivity_class
0,1,Bruceine A,CHEMBL250451,COC(=O)[C@@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)...,0.004,ug.mL-1,Active
1,2,Bruceine B,CHEMBL140809,COC(=O)[C@@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)...,0.893,ug.mL-1,Active
2,3,Bruceine C,CHEMBL455759,COC(=O)[C@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)[...,0.107,ug.mL-1,Active
3,4,Bruceine D,CHEMBL477693,CC1=CC(=O)[C@@H](O)[C@]2(C)[C@H]3[C@@H](O)[C@H...,0.835,ug.mL-1,Active
4,5,Bruceine E,CHEMBL2037038,CC1=C[C@H](O)[C@@H](O)[C@]2(C)[C@H]3[C@@H](O)[...,5000.000,nM,Intermediate
...,...,...,...,...,...,...,...
76,93,Brujavanone J,NaN,NaN,100000.000,nM,Inactive
77,94,Brujavanone K,NaN,NaN,220.000,nM,Active
78,95,Brujavanone L,NaN,NaN,10000.000,nM,Intermediate
79,96,Brujavanone M,NaN,NaN,4900.000,nM,Intermediate


In [ ]:
df.to_csv("javanica datase.csv", index= False)